Программа digit.py:

In [ ]:
# распознавание цифр
from tensorflow.keras.datasets import mnist
import numpy as np
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras.layers import Dense
(x_train, y_train), (x_test, y_test) = mnist.load_data()  # загрузка выборки
# далее отображение первых 25 изображений из обучающей выборки - можно убрать
plt.figure(figsize=(10,5))
for i in range(25):
    plt.subplot(5,5,i+1)
    plt.xticks([])
    plt.yticks([])
    plt.imshow(x_train[100+i], cmap=plt.cm.binary)
 
plt.show()
# это самое главное, это сеть. здесь можно менять
model = keras.Sequential([
    keras.layers.Flatten (input_shape=(28, 28, 1)), # потому что размер изображения 28*28
    Dense(100, activation='relu'),
    Dense(10, activation='softmax')])
# print(model.summary())     # вывод структуры НС в консоль
x_train = x_train / 255 # нормализация
x_test = x_test / 255
y_train_cat = keras.utils.to_categorical(y_train, 10) # приведение к вектору из 0 и 1 (10 - размерность)
y_test_cat = keras.utils.to_categorical(y_test, 10)
model.compile(optimizer='adam',
loss='categorical_crossentropy',
metrics=['accuracy'])
model.fit(x_train, y_train_cat, batch_size=32, epochs=10, validation_split=0.2,  verbose=False) #обучение
print('*******   loss и accuracy  на тестовом наборе   ***********')
model.evaluate(x_test, y_test_cat) # Метод evaluate прогоняет все тестовое множество и вычисляет значение критерия качества и метрики
print('*********    *****************')
print ('проверка правильности некоторой цифры')
n = 5  # изображение - это номер в наборе данных
x = np.expand_dims(x_test[n], axis=0)
# выведем рукописную цифру с заданным номером, полученное сетью значение - в виде числа и в виде вектора
res = model.predict(x)
print ('печать выходного вектора для', y_test[n])
print( res  )
print 
print( np.argmax(res) , '  - такая цифра должна быть на рисунке. Верно?')
plt.imshow(x_test[n], cmap=plt.cm.binary)
plt.show()
# выделение неверных результатов
pred = model.predict(x_test)
pred = np.argmax(pred, axis=1) # массив предсказанных результатов (на тестовом наборе)

print('вывод некоторых ошибочных некоторых результатов')
print('смотрим, правильные ли ошибочные картинки, хорошо ли нарисованы')

er=0
for i in range(10000): # проверяем тестовой набор
     if pred[i]!=y_test[i]:
         er=er+1 # считаем ошибки
         if er>10 and er<16: # смотрим 5 картинок неверно определенных сетью
             print("Значение сети: ", y_test[i], ' предсказано   ', pred[i] )
             plt.imshow(x_test[i], cmap=plt.cm.binary)
             plt.show()
        
print ('k-vo  error ', er)

Программа drop.py:

In [ ]:
# -*- coding: utf-8 -*-
"""drop.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1A9hI80SDffWH60rkBT8UDVcpPR_KiPl0
"""

import os
os.environ['TF_CPP_MIN_LOG_LEVEL']='2'
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.datasets import mnist
from tensorflow import keras
from tensorflow.keras.layers import Dense, Flatten, Dropout
(x_train, y_train), (x_test, y_test) = mnist.load_data()  # загрузка выборки
x_train = x_train / 255 # нормализация
x_test = x_test / 255
y_train_cat = keras.utils.to_categorical(y_train, 10) # приведение к вектору из 0 и 1 (10 - размерность)
y_test_cat = keras.utils.to_categorical(y_test, 10)

#Уменьшим число начальных данных
limit=4000
kn=10
x_train_data=x_train[:limit]
y_train_data=y_train_cat[:limit]

x_valid=x_train[limit:limit*2]
y_valid=y_train_cat[limit:limit*2]

model = keras.Sequential([
    Flatten(input_shape=(28, 28, 1)),
    Dense(kn , activation='relu'),
    Dense(10, activation='softmax') ])
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])
print('обучение')
his=model.fit(x_train_data, y_train_data,   epochs=50,batch_size=32, validation_data=(x_valid, y_valid),verbose=False )
model.evaluate(x_test, y_test_cat,verbose=False )
# Метод evaluate прогоняет все тестовое множество и вычисляет значение критерия качества и метрики
plt.plot(his.history['loss'])
plt.plot(his.history['val_loss'])
plt.show()

# выделение неверных результатов
pred = model.predict(x_test)
pred = np.argmax(pred, axis=1)

mask = pred == y_test
# подсчет ошибок
er=0
for k in mask:
   if k==0:
       er+=1
print('kn=', kn,'  limit=', limit,'  k-vo  error ', er)

"""Увеличим число нейронов

"""

kn=300
model = keras.Sequential([
    Flatten(input_shape=(28, 28, 1)),
    Dense(kn , activation='relu'),
    Dense(10, activation='softmax') ])
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])
print('обучение')
his=model.fit(x_train_data, y_train_data,   epochs=50,batch_size=32, validation_data=(x_valid, y_valid),verbose=False )
model.evaluate(x_test, y_test_cat,verbose=False )
# Метод evaluate прогоняет все тестовое множество и вычисляет значение критерия качества и метрики
plt.plot(his.history['loss'])
plt.plot(his.history['val_loss'])
plt.show()

pred = model.predict(x_test)
pred = np.argmax(pred, axis=1)
mask = pred == y_test
er=0
for k in mask:
   if k==0:
       er+=1
print('kn=', kn,'  limit=', limit,'  k-vo  error ', er)

"""Добавим dropout

"""

drop=0.9
model = keras.Sequential([
    Flatten(input_shape=(28, 28, 1)),
    Dense(kn , activation='relu'),
    Dropout(drop),
    Dense(10, activation='softmax') ])
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])
print('обучение')
his=model.fit(x_train_data, y_train_data,   epochs=100,batch_size=32, validation_data=(x_valid, y_valid),verbose=False )
model.evaluate(x_test, y_test_cat,verbose=False )
# Метод evaluate прогоняет все тестовое множество и вычисляет значение критерия качества и метрики
plt.plot(his.history['loss'])  # синий график
plt.plot(his.history['val_loss']) # бежевый график
plt.show()

pred = model.predict(x_test)
pred = np.argmax(pred, axis=1)
mask = pred == y_test
er=0
for k in mask:
   if k==0:
       er+=1
print('kn=', kn,'  limit=', limit, 'drop= ', drop,'  k-vo  error ', er)

Программа line.py:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras.layers import Dense
print ('Какое-то время оно думает - надо подождать. Потом построится график - его надо посмотреть и закрыть')

# обучающее множество - список входов и соответствующих выходов
c = np.array([-3, -1, 0, 2,  4, 5, 6])   # вход
f = np.array([11, 7, 5, 1, -3, -5,  -7]) # выход   y=-2x+5

model = keras.Sequential()  # слои вид сети - последовательные, друг за другом 
model.add(Dense(units=1, input_shape=(1,), activation='linear'))

# Dense - конструктор, формирование полносвязного слоя
# add - добавление слоя с заданынми параметрами
# units - количество нейронов
# input_shape=(1,) - один вход
# activation - функция активации
# Конструктор Dense формирует полносвязный слой
#    (все входы будут связаны со всеми нейронами данного слоя.
# Связи - веса и дополнительно для каждого нейрона добавляется смещение – bias. 
print ('модель построена')
model.summary()
model.compile(loss='mean_squared_error', optimizer=keras.optimizers.Adam(0.1))
# выбираем минимум среднего квадрата ошибки и оптимизацию по Adam
log = model.fit(c, f, epochs=200, verbose=False)
# fit - метод для обучения (выход, выход, к-во эпох, печатать ли текущую информацию

plt.plot(log.history['loss'])
plt.grid(True)
plt.show()
# график  показывает, как меняется loss - изменить количество эпох


print('f(2)=',model.predict(np.array([2])))  # печать предсказания модели для 2
print('f(-10)=',model.predict(np.array([-10])))
print('f(10)=',model.predict(np.array([10])))
print('веса  ', model.get_weights())  # при хорошей модели веса равны коэффициентам

Программа raspozn_cifr_svertka.py:

In [ ]:
# распознавание цифр
from tensorflow.keras.datasets import mnist
import numpy as np
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras.layers import Dense,  Flatten, Dropout, Conv2D, MaxPooling2D
(x_train, y_train), (x_test, y_test) = mnist.load_data()  # загрузка выборки

model = keras.Sequential([
    Conv2D(32, (3,3), padding='same', activation='relu', input_shape=(28, 28, 1)),
    MaxPooling2D((2, 2), strides=2),
    Conv2D(64, (3,3), padding='same', activation='relu'),
    MaxPooling2D((2, 2), strides=2),
    Flatten(),
    Dense(128, activation='relu'),
    Dense(10,  activation='softmax')
])
print(model.summary())     # вывод структуры НС в консоль
x_train = x_train / 255 # нормализация
x_test = x_test / 255
y_train_cat = keras.utils.to_categorical(y_train, 10) # приведение к вектору из 0 и 1 (10 - размерность)
y_test_cat = keras.utils.to_categorical(y_test, 10)
#для сверточной НС множества x_tarin и x_test нужно дополнительно подготовить.
#на входе такой сети ожидается четырехмерный теyзор,
# нужно  добавить еще одно измерение  для цветовой компоненты
x_train = np.expand_dims(x_train, axis=3)
x_test = np.expand_dims(x_test, axis=3)
print( x_train.shape )
model.compile(optimizer='adam',
loss='categorical_crossentropy',
metrics=['accuracy'])
hit=model.fit(x_train, y_train_cat, batch_size=32, epochs=5, validation_split=0.2,  verbose=False) #обучение
#print('*******   loss и accuracy  на тестовом наборе   ***********')
#model.evaluate(x_test, y_test_cat) # Метод evaluate прогоняет все тестовое множество и вычисляет значение критерия качества и метрики
#print('*********    *****************')
print ('проверка правильности некоторой цифры')
n = 5  # изображение - это номер в наборе данных
x = np.expand_dims(x_test[n], axis=0)
# выведем рукописную цифру с заданным номером, полученное сетью значение - в виде числа и в виде вектора
res = model.predict(x)
print ('печать выходного вектора для', y_test[n])
print( res  )
print 
print( np.argmax(res) , '  - такая цифра должна быть на рисунке. Верно?')
plt.imshow(x_test[n], cmap=plt.cm.binary)
plt.show()
# выделение неверных результатов
pred = model.predict(x_test)
pred = np.argmax(pred, axis=1) # массив предсказанных результатов (на тестовом наборе)

print('вывод некоторых ошибочных некоторых результатов')
print('смотрим, правильные ли ошибочные картинки, хорошо ли нарисованы')

er=0
for i in range(10000): # проверяем тестовой набор
     if pred[i]!=y_test[i]:
         er=er+1 # считаем ошибки
         if er>10 and er<16: # смотрим 5 картинок неверно определенных сетью
             print("Значение сети: ", y_test[i], ' предсказано   ', pred[i] )
             plt.imshow(x_test[i], cmap=plt.cm.binary)
             plt.show()
        
print ('k-vo  error ', er)

Программа rnn_words.py:

In [ ]:
# рекурентная сеть прогнозирование следующего слова по трем предыдущим
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np

from tensorflow.keras.layers import Dense, SimpleRNN, Input, Embedding
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.text import Tokenizer, text_to_word_sequence
from tensorflow.keras.utils import to_categorical

with open('cat.txt', 'r', encoding='utf-8') as f:  # файл
    texts = f.read()
    texts = texts.replace('\ufeff', '')  # убираем первый невидимый символ

maxWordsCount = 1500
tokenizer = Tokenizer(num_words=maxWordsCount, filters='!–"—#$%&amp;()*+,-./:;<=>?@[\\]^_`{|}~\t\n\r«»',
                      lower=True, split=' ', char_level=False)
tokenizer.fit_on_texts([texts])

dist = list(tokenizer.word_counts.items())
print(dist[:40])

data = tokenizer.texts_to_sequences([texts])
# res = to_categorical(data[0], num_classes=maxWordsCount)
# print(res.shape)
res = np.array( data[0] )

inp_words = 3
n = res.shape[0] - inp_words

X = np.array([res[i:i + inp_words] for i in range(n)])
Y = to_categorical(res[inp_words:], num_classes=maxWordsCount)

model = Sequential()
model.add(Embedding(maxWordsCount, 256, input_length = inp_words))
model.add(SimpleRNN(128, activation='tanh'))
model.add(Dense(maxWordsCount, activation='softmax'))
model.summary()

model.compile(loss='categorical_crossentropy', metrics=['accuracy'], optimizer='adam')

history = model.fit(X, Y, batch_size=32, epochs=40, verbose=False)


def buildPhrase(texts, str_len=20):
    res = texts
    data = tokenizer.texts_to_sequences([texts])[0]
    for i in range(str_len):
        # x = to_categorical(data[i: i + inp_words], num_classes=maxWordsCount)  # преобразуем в One-Hot-encoding
        # inp = x.reshape(1, inp_words, maxWordsCount)
        x = data[i: i + inp_words]
        inp = np.expand_dims(x, axis=0)

        pred = model.predict(inp, verbose=False)
        indx = pred.argmax(axis=1)[0]
        data.append(indx)

        res += " " + tokenizer.index_word[indx]  # дописываем строку

    return res


res = buildPhrase("добрый роман для")
print(res)
print("*** * ** * ***** ***** ")
res = buildPhrase("рукой престрастной так")
print(res)
print("*** * ** * ***** ***** ")
res = buildPhrase("прими святой живой")
print(res)
print("*** * ** * ***** ***** ")
res = buildPhrase("горый ответ тебе")
print(res)


Программа rnn.py

In [ ]:
# рекурентная сеть прогноз букв
import os
os.environ['TF_CPP_MIN_LOG_LEVEL']='2'

import numpy as np
import re
from tensorflow.keras.layers import Dense, SimpleRNN, Input
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.text import Tokenizer

with open ('cat.txt', 'r', encoding='utf-8') as f:
           text=f.read()
           text=text.replace('\ufeff', '')# убираем первый невидимый символ
           text=re.sub(r'[^А-я]', ' ', text) # заменяем все символы кроме русских букв на пустоту
           text=text.replace('  ', ' ')
           text=text.lower()
# парсим текст как последовательность символов
num_characters = 35 # 33 бувы и пробел
tokenizer = Tokenizer(num_words=num_characters, char_level=True)
tokenizer.fit_on_texts(text)
print(tokenizer.word_index) # сформирован словарь

inp_chars = 3
data = tokenizer.texts_to_matrix(text)
n = data.shape[0]-inp_chars
X = np.array([data[i:i+inp_chars, :] for i in range(n)])
Y = data[inp_chars:] #предсказание следующего символа

model = Sequential()
model.add(Input((inp_chars, num_characters))) 
model.add(SimpleRNN(500, activation='tanh')) 
model.add(Dense(num_characters, activation='softmax'))
model.summary()

model.compile(loss='categorical_crossentropy', metrics=['accuracy'], optimizer='adam')
history = model.fit(X, Y, batch_size=64, epochs=50,  verbose=False )

def buildPhrase(inp_str, str_len = 50):
  for i in range(str_len):
    x = []
    for j in range(i, i+inp_chars):
      x.append(tokenizer.texts_to_matrix(inp_str[j])) # преобразуем символы в One-Hot-encoding
 
    x = np.array(x)
    inp = x.reshape(1, inp_chars, num_characters)
 
    pred = model.predict( inp, verbose=False ) # предсказываем OHE четвертого символа
    d = tokenizer.index_word[pred.argmax(axis=1)[0]] # получаем ответ в символьном представлении
 
    inp_str += d # дописываем строку
 
  return inp_str

for mm in ("дом", "вор", "кот", "дуб", "лес", "вой", "гов", "мол", "ход" ):
    res = buildPhrase(mm)
    print(res)

Програма rnn0.py:

In [ ]:
# рекурентная сеть прогноз букв
import os
os.environ['TF_CPP_MIN_LOG_LEVEL']='2'

import numpy as np
import re
from tensorflow.keras.layers import Dense, SimpleRNN, Input
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.text import Tokenizer

with open ('cat.txt', 'r', encoding='utf-8') as f:
           text=f.read()
           text=text.replace('\ufeff', '')# убираем первый невидимый символ
           text=re.sub(r'[^А-я]', ' ', text) # заменяем все символы кроме русских букв на пустоту
           text=text.replace('  ', ' ')
           text=text.lower()
# парсим текст как последовательность символов
num_characters = 35 # 33 бувы и пробел
tokenizer = Tokenizer(num_words=num_characters, char_level=True)
tokenizer.fit_on_texts(text)
print(tokenizer.word_index) # сформирован словарь

inp_chars = 3
data = tokenizer.texts_to_matrix(text)
n = data.shape[0]-inp_chars
X = np.array([data[i:i+inp_chars, :] for i in range(n)])
Y = data[inp_chars:] #предсказание следующего символа

model = Sequential()
model.add(Input((inp_chars, num_characters))) 
model.add(SimpleRNN(500, activation='tanh')) 
model.add(Dense(num_characters, activation='softmax'))
model.summary()

model.compile(loss='categorical_crossentropy', metrics=['accuracy'], optimizer='adam')
history = model.fit(X, Y, batch_size=64, epochs=50,  verbose=False )

def buildPhrase(inp_str, str_len = 50):
  for i in range(str_len):
    x = []
    for j in range(i, i+inp_chars):
      x.append(tokenizer.texts_to_matrix(inp_str[j])) # преобразуем символы в One-Hot-encoding
 
    x = np.array(x)
    inp = x.reshape(1, inp_chars, num_characters)
 
    pred = model.predict( inp, verbose=False ) # предсказываем OHE четвертого символа
    d = tokenizer.index_word[pred.argmax(axis=1)[0]] # получаем ответ в символьном представлении
 
    inp_str += d # дописываем строку
 
  return inp_str

for mm in ("дом", "вор", "кот", "дуб", "лес", "вой", "гов", "мол", "ход" ):
    res = buildPhrase(mm)
    print(res)